In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone Code từ GitHub
%cd /content
!rm -rf Forget-MI-LoKU
!git clone https://github.com/nhnhu146/Forget-MI-LoKU.git
%cd Forget-MI-LoKU

# 3. Cài đặt thư viện
!pip install -q pydicom scikit-image wandb pyyaml pandas
!pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"

print("✅ Môi trường và mã nguồn đã sẵn sàng!")

In [ ]:
# 4. Giải nén Dữ liệu & Models (Bản Fix lỗi mv: failed to access)
import os
import shutil

DRIVE_PATH = "/content/drive/MyDrive/Forget-MI-Project"

def robust_unzip_python(zip_name, target_dir):
    zip_full_path = os.path.join(DRIVE_PATH, zip_name)
    if not os.path.exists(zip_full_path):
        print(f"❌ Không tìm thấy file {zip_name} trên Drive!")
        return
    
    print(f"📦 Đang xử lý {zip_name}...")
    temp_extract = "/tmp/extract_temp"
    if os.path.exists(temp_extract): shutil.rmtree(temp_extract)
    os.makedirs(temp_extract)
    
    # Giải nén bằng lệnh shell
    !unzip -q -o {zip_full_path} -d {temp_extract}
    
    # Tạo thư mục đích sạch sẽ
    os.makedirs(target_dir, exist_ok=True)
    
    # Duyệt tất cả file và di chuyển bằng Python (An toàn hơn mv shell)
    count = 0
    for root, dirs, files in os.walk(temp_extract):
        for f in files:
            src = os.path.join(root, f)
            dst = os.path.join(target_dir, f)
            # Nếu file đã tồn tại ở đích, ghi đè
            if os.path.exists(dst): os.remove(dst)
            shutil.move(src, dst)
            count += 1
            
    shutil.rmtree(temp_extract)
    print(f"✅ Đã di chuyển {count} file vào: {target_dir}")

# Thực hiện giải nén (Data giải nén vào gốc để giữ cấu trúc con nếu có)
# Tuy nhiên để an toàn cho ảnh, ta vẫn dùng hàm này cho Model
robust_unzip_python("base_model.zip", "./forgetme/training_original_model")
robust_unzip_python("retrained_model.zip", "./model_retrained_3per")

# Riêng Data.zip, nếu nó chứa cả folder con 'img_data', 'metadata' thì giải nén vào gốc repo
print("📦 Đang giải nén data.zip...")
!unzip -q -o {DRIVE_PATH}/data.zip -d ./

print("\n🚀 Tất cả Dữ liệu & Model đã sẵn sàng!")

In [ ]:
# 5. Tiền xử lý dữ liệu (Tạo all_data.tsv)
!python make_tsv.py

# 6. Thiết lập thư mục lưu kết quả bền vững trên Drive
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
import os
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Tạo liên kết Symlink
if os.path.exists("unlearning_output"):
    !rm -rf unlearning_output
!ln -s {DRIVE_RESULTS} ./unlearning_output

print(f"✅ Đã kết nối thư mục Output với Drive: {DRIVE_RESULTS}")

In [ ]:
# 7. Chạy Unlearning LoKU
!PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py --config config.yaml